# Get Yahoo Public Picks

Used to compare my picks with general public concensus. Assumes yahoo data (https://tournament.fantasysports.yahoo.com/womens-basketball-bracket/pickdistribution?year=2024) has been exported to Excel. 

Remember that you may have to manually update play-ins in the code.

In [1]:
season = 2025

play_in_updates = [
    # (yahoo_name, corrected_name) for every play-in matchup
    ('UCSD/SOU', 'UC San Diego'),
    ('HP/WM', 'High Point'),
    ('ISU/PRIN', 'Iowa State'),
    ('COL/WASH', 'Washington'),
]

season, play_in_updates

(2025,
 [('UCSD/SOU', 'UC San Diego'),
  ('HP/WM', 'High Point'),
  ('ISU/PRIN', 'Iowa State'),
  ('COL/WASH', 'Washington')])

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.concat(
    [
        pd.read_excel(fr'..\data\unprocessed\womens_yahoo\yahoo_picks_{season}.xlsx', sheet_name=f'round_{round_}')
        .assign(Round=round_)
        for round_ in range(1, 7)
    ],
    ignore_index=True,
)

df['Seed'] = df['Team (Seed)'].str.extract(r'\((\d+)\)').astype(int)
df['Team (Seed)'] = df['Team (Seed)'].str.replace(r'\(\d+\)', '', regex=True)

df.rename(columns={'Team (Seed)': 'Team'}, inplace=True)

df = df.pivot(index=['Team', 'Seed'], columns=['Round'], values='% Picked').reset_index()
df.columns = ['Team', 'Seed', 'Round 1', 'Round 2', 'Round 3', 'Round 4', 'Round 5', 'Round 6']

df

,Team,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,Alabama,5,0.8779,0.3971,0.0445,0.0158,0.0069,0.0031
1,Arkansas St.,15,0.0322,0.0148,0.0070,0.0034,0.0012,0.0005
2,Ball St.,12,0.1248,0.0281,0.0084,0.0041,0.0020,0.0012
3,Baylor,4,0.9077,0.6385,0.0903,0.0439,0.0108,0.0040
4,COL/WASH,11,0.1361,0.0245,0.0084,0.0034,0.0015,0.0007
...,...,...,...,...,...,...,...,...
59,USC,1,0.9665,0.9211,0.8259,0.4419,0.2128,0.1182
60,Utah,8,0.4881,0.0277,0.0124,0.0062,0.0027,0.0014
61,Vanderbilt,7,0.5852,0.1246,0.0288,0.0078,0.0023,0.0009
62,Vermont,15,0.0425,0.0157,0.0075,0.0029,0.0006,0.0003


Fix play-ins

In [3]:
df.loc[df['Team'].str.contains('/', regex=False), :]

,Team,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
4,COL/WASH,11,0.1361,0.0245,0.0084,0.0034,0.0015,0.0007
17,HP/WM,16,0.0232,0.0102,0.0050,0.0026,0.0009,0.0003
19,ISU/PRIN,11,0.0820,0.0170,0.0064,0.0030,0.0014,0.0005
58,UCSD/SOU,16,0.0220,0.0105,0.0051,0.0029,0.0013,0.0006


In [4]:
for yahoo_name, corrected_name in play_in_updates:
    df.loc[df['Team'] == yahoo_name, 'Team'] = corrected_name

df.loc[df['Team'].str.contains('/', regex=False), :]

,Team,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6


Map with Kaggle data

In [5]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\WNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2025,1,W,False,3376
1,2025,2,W,False,3181
2,2025,3,W,False,3314
3,2025,4,W,False,3268
4,2025,5,W,False,3104
...,...,...,...,...,...
63,2025,12,Z,False,3193
64,2025,13,Z,False,3251
65,2025,14,Z,False,3195
66,2025,15,Z,False,3117


In [6]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\WTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

# df_spellings.loc[df_spellings.shape[0]] = ['fdu', 3192]
df_spellings.loc[df_spellings.shape[0]] = ['sdsu', 3361]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,3394
1,a&m-corpus christi,3394
2,abilene chr,3101
3,abilene christian,3101
4,abilene-christian,3101
...,...,...
1171,youngstown st.,3464
1172,youngstown state,3464
1173,youngstown-st,3464
1174,youngstown-state,3464


In [7]:
df_spellings = pd.merge(
    df_spellings,
    df_seeds[['TeamID', 'Seed']],
    how='inner',
    on=['TeamID']
)

df_spellings

,TeamNameSpelling,TeamID,Seed
0,alabama,3104,5
1,arkansas st,3117,15
2,arkansas st.,3117,15
3,arkansas state,3117,15
4,arkansas-st,3117,15
...,...,...,...
182,william-mary,3456,16
183,wis.-green bay,3453,12
184,wisconsin-green bay,3453,12
185,wisconsin-green-bay,3453,12


In [8]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

team_spellings = df_spellings['TeamNameSpelling'].unique()
yahoo_teams = df.loc[~df['Team'].str.contains('^playin', regex=True), 'Team'].unique()

df_match = pd.DataFrame(
    [
        [
            yahoo_team,
            *process.extract(
                yahoo_team,
                team_spellings,
                scorer=token_sort_ratio,
                limit=1
            )[0][:2]
        ] for yahoo_team in tqdm(yahoo_teams)
    ],
    columns=['Yahoo Team', 'Team Spelling', 'Match Score']
).sort_values('Match Score', ignore_index=True)

df_match.head(25)

C:\Users\mhugh\AppData\Local\Temp\ipykernel_11416\794795546.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/64 [00:00<?, ?it/s]

,Yahoo Team,Team Spelling,Match Score
0,N. Carolina,north carolina,83
1,S.F. Austin,sf austin,84
2,N.C. Greensboro,nc greensboro,89
3,Fla Gulf Coast,fl gulf coast,96
4,Alabama,alabama,100
5,Montana St.,montana st,100
6,Murray St.,murray st,100
7,N.C. State,n.c. state,100
8,Nebraska,nebraska,100
9,Norfolk St.,norfolk st,100


In [9]:
yahoo_to_spelling = dict(zip(df_match['Yahoo Team'], df_match['Team Spelling']))
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

df.insert(1, 'TeamID', df['Team'].map(yahoo_to_spelling).map(spelling_to_id))

df

,Team,TeamID,Seed,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,Alabama,3104,5,0.8779,0.3971,0.0445,0.0158,0.0069,0.0031
1,Arkansas St.,3117,15,0.0322,0.0148,0.0070,0.0034,0.0012,0.0005
2,Ball St.,3123,12,0.1248,0.0281,0.0084,0.0041,0.0020,0.0012
3,Baylor,3124,4,0.9077,0.6385,0.0903,0.0439,0.0108,0.0040
4,Washington,3449,11,0.1361,0.0245,0.0084,0.0034,0.0015,0.0007
...,...,...,...,...,...,...,...,...,...
59,USC,3425,1,0.9665,0.9211,0.8259,0.4419,0.2128,0.1182
60,Utah,3428,8,0.4881,0.0277,0.0124,0.0062,0.0027,0.0014
61,Vanderbilt,3435,7,0.5852,0.1246,0.0288,0.0078,0.0023,0.0009
62,Vermont,3436,15,0.0425,0.0157,0.0075,0.0029,0.0006,0.0003


In [10]:
import numpy as np

id_to_playin = dict(zip(df_seeds['TeamID'], df_seeds['Play In']))
id_to_playin[np.nan] = True

df.insert(3, 'Play In', df['TeamID'].map(id_to_playin))

df

,Team,TeamID,Seed,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,Alabama,3104,5,False,0.8779,0.3971,0.0445,0.0158,0.0069,0.0031
1,Arkansas St.,3117,15,False,0.0322,0.0148,0.0070,0.0034,0.0012,0.0005
2,Ball St.,3123,12,False,0.1248,0.0281,0.0084,0.0041,0.0020,0.0012
3,Baylor,3124,4,False,0.9077,0.6385,0.0903,0.0439,0.0108,0.0040
4,Washington,3449,11,True,0.1361,0.0245,0.0084,0.0034,0.0015,0.0007
...,...,...,...,...,...,...,...,...,...,...
59,USC,3425,1,False,0.9665,0.9211,0.8259,0.4419,0.2128,0.1182
60,Utah,3428,8,False,0.4881,0.0277,0.0124,0.0062,0.0027,0.0014
61,Vanderbilt,3435,7,False,0.5852,0.1246,0.0288,0.0078,0.0023,0.0009
62,Vermont,3436,15,False,0.0425,0.0157,0.0075,0.0029,0.0006,0.0003


In [11]:
id_to_region = dict(zip(df_seeds['TeamID'], df_seeds['Region']))

df.insert(3, 'Region', df['TeamID'].map(id_to_region))

df

,Team,TeamID,Seed,Region,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,Alabama,3104,5,W,False,0.8779,0.3971,0.0445,0.0158,0.0069,0.0031
1,Arkansas St.,3117,15,Z,False,0.0322,0.0148,0.0070,0.0034,0.0012,0.0005
2,Ball St.,3123,12,Y,False,0.1248,0.0281,0.0084,0.0041,0.0020,0.0012
3,Baylor,3124,4,Y,False,0.9077,0.6385,0.0903,0.0439,0.0108,0.0040
4,Washington,3449,11,W,True,0.1361,0.0245,0.0084,0.0034,0.0015,0.0007
...,...,...,...,...,...,...,...,...,...,...,...
59,USC,3425,1,Z,False,0.9665,0.9211,0.8259,0.4419,0.2128,0.1182
60,Utah,3428,8,W,False,0.4881,0.0277,0.0124,0.0062,0.0027,0.0014
61,Vanderbilt,3435,7,W,False,0.5852,0.1246,0.0288,0.0078,0.0023,0.0009
62,Vermont,3436,15,Y,False,0.0425,0.0157,0.0075,0.0029,0.0006,0.0003


In [12]:
df.loc[df['Region'].isna(), :]

,Team,TeamID,Seed,Region,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6


Redistribute play-in probabilities to the teams that won

If using 2023, it is unclear which play-in teams are which

In [13]:
# for seed in df.loc[df['Play In'], 'Seed'].unique():
#     df.loc[
#         (~df['Team'].str.contains('^playin', regex=True)) & 
#         (df['Seed'] == seed) &
#         (df['Play In']), 
#         [f'Round {i}' for i in range(1, 7)]
#     ] += df.loc[
#         (df['Team'].str.contains('^playin', regex=True)) & 
#         (df['Seed'] == seed) &
#         (df['Play In']), 
#         [f'Round {i}' for i in range(1, 7)]
#     ].mean(axis=0)

# df = df.loc[
#     ~df['Team'].str.contains('^playin', regex=True), 
#     :
# ].reset_index(drop=True)

# df['TeamID'] = df['TeamID'].astype(int)

# df.loc[df['Play In'], :]

Redistribute percentages to account for rounding inaccuracies

In [14]:
# for i in range(1, 7):
#     df[f'Round {i}'] = df[f'Round {i}'] / df[f'Round {i}'].sum() * 2**(6 - i)

# df

In [15]:
df.sum()

Team       AlabamaArkansas St.Ball St.BaylorWashingtonCal...
TeamID                                                210734
Seed                                                     544
Region     WZYYWZZXWXZZYYYYWXYXXWZZZYWZXWXYYZXZWZYXWXXZZW...
Play In                                                    4
Round 1                                              31.6412
Round 2                                              15.8613
Round 3                                                7.941
Round 4                                               3.9748
Round 5                                               1.9987
Round 6                                                  1.0
dtype: object

In [16]:
df.insert(df.columns.get_loc('Seed'), 'Region Seed', df['Region'] + df['Seed'].astype(str).str.zfill(2))

df

,Team,TeamID,Region Seed,Seed,Region,Play In,Round 1,Round 2,Round 3,Round 4,Round 5,Round 6
0,Alabama,3104,W05,5,W,False,0.8779,0.3971,0.0445,0.0158,0.0069,0.0031
1,Arkansas St.,3117,Z15,15,Z,False,0.0322,0.0148,0.0070,0.0034,0.0012,0.0005
2,Ball St.,3123,Y12,12,Y,False,0.1248,0.0281,0.0084,0.0041,0.0020,0.0012
3,Baylor,3124,Y04,4,Y,False,0.9077,0.6385,0.0903,0.0439,0.0108,0.0040
4,Washington,3449,W11,11,W,True,0.1361,0.0245,0.0084,0.0034,0.0015,0.0007
...,...,...,...,...,...,...,...,...,...,...,...,...
59,USC,3425,Z01,1,Z,False,0.9665,0.9211,0.8259,0.4419,0.2128,0.1182
60,Utah,3428,W08,8,W,False,0.4881,0.0277,0.0124,0.0062,0.0027,0.0014
61,Vanderbilt,3435,W07,7,W,False,0.5852,0.1246,0.0288,0.0078,0.0023,0.0009
62,Vermont,3436,Y15,15,Y,False,0.0425,0.0157,0.0075,0.0029,0.0006,0.0003


In [17]:
df.to_parquet(f'../data/preprocessed/womens_yahoo/yahoo_picks_{season}.parquet')

'Done'

'Done'